
# mHealth Privacy Metrics Calculation

This notebook calculates the privacy metrics of mHealth apps:

- **ADII**: App Data Invasiveness Index
- **DGI**: Disclosure Gap Index
- **PCLR**: Pre-Consent Leakage Rate
- **AS**: Adaptation Score

In [ ]:
from pathlib import Path
import ast
import math
import re
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D


mhealth_apps_path = Path("../data/mhealth_apps_filled.csv")
traffic_results_path = Path("../data/traffic_results_detailed.csv")

df_apps = pd.read_csv(mhealth_apps_path)
df_traffic = pd.read_csv(traffic_results_path)

REGION_MAP = {
    "us": "North America", "ca": "North America", "gl":"North America",
    "mx": "Latin America", "br": "Latin America", "ar": "Latin America", 
    "cl": "Latin America",
    "de": "Europe", "fr": "Europe", "gb": "Europe", 
    "pl": "Europe",
    "tr": "Middle East", "il": "Middle East", "sa": "Middle East", 
    "ae": "Middle East",
    "in": "Asia", "jp": "Asia", "sg": "Asia", 
    "kr": "Asia",
    "za": "Africa", "mz": "Africa", "ng": "Africa", 
    "ke": "Africa",
    "au": "Oceania", "nz": "Oceania", 
    "pg": "Oceania"
}

COUNTRY_LABEL_MAP = {
    "us": "United States", "ca": "Canada", "gl":"Greenland",
    "mx": "Mexico", "br": "Brazil", "ar": "Argentina",
    "cl": "Chile",
    "de": "Germany", "fr": "France", "gb": "Great Britain",
    "pl": "Poland",
    "tr": "Turkey", "il": "Israel", "sa": "Saudi Arabia",
    "ae": "United Arab Emirates",
    "in": "India", "jp": "Japan", "sg": "Singapore",
    "kr": "South Korea",
    "za": "South Africa", "mz": "Mozambique", "ng": "Nigeria",
    "ke": "Kenya",
    "au": "Australia", "nz": "New Zealand",
    "pg": "Papua New Guinea"
}

REGION_ORDER = [
    "North America", "Latin America", "Europe",
    "Middle East", "Asia", "Africa", "Oceania"
]

METRIC_COLORS = {
    "ADII": "#1f77b4",
    "DGI": "#d62728",
    "PCLR": "#2ca02c",
    "AS": "#9467bd",
}

print("mhealth_apps shape:", df_apps.shape)
print("traffic_results shape:", df_traffic.shape)

if "countries" in df_apps.columns:
    df_apps = df_apps.drop(columns=["countries"])
    print("'countries' column dropped.")

assert "app_id" in df_apps.columns, "app_id missing in mhealth_apps"
assert "app_id" in df_traffic.columns, "app_id missing in traffic_results"

# MERGE / FILTER
base_df = df_traffic.merge(df_apps, on="app_id", how="left")
print("Merged dataset shape:", base_df.shape)

base_df["country"] = (
    base_df["country"]
    .astype(str)
    .str.lower()
    .str.strip()
)

base_df["region"] = base_df["country"].map(REGION_MAP)
base_df["region"] = base_df["region"].fillna("Other")

base_df = base_df[
    base_df["country"].isin(COUNTRY_LABEL_MAP.keys())
].copy()

base_df["country_label"] = base_df["country"].map(COUNTRY_LABEL_MAP)
base_df["region"] = base_df["country"].map(REGION_MAP)

print("Filtered dataset shape:", base_df.shape)

# SAFETY GUARD: this cell used to unconditionally overwrite
# ../data/mhealth_apps_traffic.csv on every run. That file is the shared,
# canonical input every other notebook in this pipeline reads -- but this
# cell's own re-merge does not reproduce it exactly (verified: re-running
# this cell changes downstream ADII from a mean of ~931 to ~841 and DGI
# from ~0.83 to ~0.89, even though row/app/country counts stay the same),
# which points to this cell using a different country/region mapping than
# whatever process originally produced the canonical file. Concretely,
# this has now silently corrupted mhealth_apps_traffic.csv from routine
# notebook execution (not from anything this cell's own logic intended to
# change). Until that discrepancy is root-caused, this write is guarded so
# an accidental "Run All" cannot silently clobber the canonical file. To
# regenerate it deliberately, delete the existing file first (or change
# FORCE_OVERWRITE_TRAFFIC_CSV below), and be aware the regenerated version
# has been observed to differ from the current one.
FORCE_OVERWRITE_TRAFFIC_CSV = False

_traffic_out_path = Path("../data/mhealth_apps_traffic.csv")
if _traffic_out_path.exists() and not FORCE_OVERWRITE_TRAFFIC_CSV:
    print(
        f"SKIPPED writing {_traffic_out_path}: file already exists and "
        "FORCE_OVERWRITE_TRAFFIC_CSV is False. `base_df` computed above is "
        "NOT what downstream cells/notebooks use -- they read the existing "
        "file from disk. See the comment above this cell for why."
    )
else:
    base_df.to_csv(_traffic_out_path, index=False)
    print(f"Wrote {_traffic_out_path}. Final dataset shape:", base_df.shape)


In [ ]:
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 200
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

metrics_results = Path("../data/mhealth_apps_traffic.csv")
if not metrics_results.exists():
    raise FileNotFoundError(
        f"Could not find dataset at {metrics_results.resolve()}. "
        "Update `metrics_results` if your file is stored elsewhere."
    )

region_palette = plt.cm.tab10(np.linspace(0, 1, len(REGION_ORDER)))
REGION_COLORS = {region: region_palette[i] for i, region in enumerate(REGION_ORDER)}
category_palette = plt.cm.tab20(np.linspace(0, 1, 20))



## 2. Load and prepare the dataset

In [ ]:
import ast
import re
from itertools import combinations

import numpy as np
import pandas as pd


def safe_literal_eval(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (list, dict, tuple, set)):
        return value
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "no content"}:
        return np.nan
    try:
        return ast.literal_eval(text)
    except Exception:
        return value


def parse_semicolon_frequency_map(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "no content"}:
        return {}
    result = {}
    for part in text.split(";"):
        part = part.strip()
        if not part or ":" not in part:
            continue
        key, val = part.rsplit(":", 1)
        key = key.strip().lower()
        val = val.strip()
        try:
            result[key] = float(val)
        except Exception:
            continue
    return result


def parse_domain_set(value):
    if pd.isna(value):
        return set()
    if isinstance(value, (list, set, tuple)):
        return {str(x).strip().lower() for x in value if str(x).strip()}
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "no content"}:
        return set()
    parts = re.split(r"[;,]\s*", text)
    return {p.strip().lower() for p in parts if p.strip()}


def coerce_numeric(series, fillna=np.nan):
    out = pd.to_numeric(series, errors="coerce")
    if fillna is not np.nan:
        out = out.fillna(fillna)
    return out


def merge_frequency_maps(*maps):
    merged = {}
    for mp in maps:
        if not isinstance(mp, dict):
            continue
        for k, v in mp.items():
            try:
                key = str(k).strip().lower()
                merged[key] = merged.get(key, 0.0) + float(v)
            except Exception:
                continue
    return merged


def build_country_feature_dict_from_df(country_df, state="both"):
    feature_dict = {}

    if state in {"pre", "both"}:
        pre_freq_maps = country_df.get("pre_type_frequencies_map", pd.Series(dtype=object))
        merged_pre = merge_frequency_maps(*pre_freq_maps.tolist())
        for k, v in merged_pre.items():
            feature_dict[f"pre_freq::{k}"] = float(v)

        pre_types = set().union(*country_df.get("pre_observed_data_types_set", pd.Series(dtype=object)).tolist()) \
            if "pre_observed_data_types_set" in country_df.columns else set()
        for t in pre_types:
            feature_dict[f"pre_type::{t}"] = 1.0

        pre_domains = set().union(*country_df.get("pre_domains_set", pd.Series(dtype=object)).tolist()) \
            if "pre_domains_set" in country_df.columns else set()
        for d in pre_domains:
            feature_dict[f"pre_domain::{d}"] = 1.0

    if state in {"post", "both"}:
        post_freq_maps = country_df.get("post_type_frequencies_map", pd.Series(dtype=object))
        merged_post = merge_frequency_maps(*post_freq_maps.tolist())
        for k, v in merged_post.items():
            feature_dict[f"post_freq::{k}"] = float(v)

        post_types = set().union(*country_df.get("post_observed_data_types_set", pd.Series(dtype=object)).tolist()) \
            if "post_observed_data_types_set" in country_df.columns else set()
        for t in post_types:
            feature_dict[f"post_type::{t}"] = 1.0

        post_domains = set().union(*country_df.get("post_domains_set", pd.Series(dtype=object)).tolist()) \
            if "post_domains_set" in country_df.columns else set()
        for d in post_domains:
            feature_dict[f"post_domain::{d}"] = 1.0

    return feature_dict


def normalized_l1_distance(feat_a, feat_b):
    keys = set(feat_a.keys()) | set(feat_b.keys())
    if not keys:
        return 0.0

    num = 0.0
    den = 0.0
    for k in keys:
        a = float(feat_a.get(k, 0.0))
        b = float(feat_b.get(k, 0.0))
        num += abs(a - b)
        den += abs(a) + abs(b)

    return 0.0 if den == 0 else num / den


# DGI helpers
INVALID_DISCLOSURE_PHRASES = {
    "not mentioned",
    "not applicable",
    "not at all",
    "none mentioned",
    "none",
    "no",
    "no content",
    "nan",
    "n/a",
    "na",
    "",
}

INVALID_DISCLOSURE_PATTERNS = [
    r"^not\s+mentioned$",
    r"^not\s+applicable$",
    r"^not\s+at\s+all$",
    r"^none\s+mentioned$",
    r"^none$",
    r"^no\s+content$",
    r"^n/?a$",
    r"^nan$",
    r".*lacks specific details on data handling and user rights.*",
]

def is_invalid_disclosure_token(token: str) -> bool:
    if token is None:
        return True

    t = str(token).strip().lower()
    if not t:
        return True

    if t in INVALID_DISCLOSURE_PHRASES:
        return True

    for pattern in INVALID_DISCLOSURE_PATTERNS:
        if re.fullmatch(pattern, t, flags=re.I):
            return True

    return False


def normalize_token(token: str) -> str:
    t = str(token).strip().lower()
    t = re.sub(r"\s+", " ", t)
    return t.strip(" '\"")


def parse_listish_to_set(value, filter_invalid=False):
    if pd.isna(value):
        return set()

    tokens = set()

    if isinstance(value, dict):
        raw_items = value.keys()
    elif isinstance(value, (list, set, tuple)):
        raw_items = value
    else:
        parsed = safe_literal_eval(value)
        if isinstance(parsed, dict):
            raw_items = parsed.keys()
        elif isinstance(parsed, (list, set, tuple)):
            raw_items = parsed
        else:
            text = str(value).strip()
            if not text or text.lower() in {"nan", "none", "no content"}:
                return set()
            raw_items = re.split(r"[;,]\s*", text)

    for item in raw_items:
        tok = normalize_token(item)
        if not tok:
            continue
        if filter_invalid and is_invalid_disclosure_token(tok):
            continue
        tokens.add(tok)

    return tokens


def union_from_columns(row, columns, filter_invalid=False):
    result = set()
    for col in columns:
        if col not in row.index:
            continue
        value = row[col]
        parsed = parse_listish_to_set(value, filter_invalid=filter_invalid)
        result |= parsed
    return result


def compute_dgi_row(row, observed_columns, disclosure_columns):
    observed = union_from_columns(row, observed_columns, filter_invalid=False)
    disclosed = union_from_columns(row, disclosure_columns, filter_invalid=True)

    missing = observed - disclosed
    misleading = disclosed - observed

    dgi = np.nan if len(observed) == 0 else len(missing) / len(observed)

    return pd.Series({
        "observed_set_final": sorted(observed),
        "disclosed_set_final": sorted(disclosed),
        "missing_set": sorted(missing),
        "misleading_set": sorted(misleading),
        "observed_count": len(observed),
        "disclosed_count": len(disclosed),
        "missing_count": len(missing),
        "misleading_count": len(misleading),
        "DGI": dgi,
    })


def compute_app_level_as(app_df):
    if "country" not in app_df.columns:
        return np.nan

    app_df = app_df.dropna(subset=["country"]).copy()
    if app_df["country"].nunique() < 2:
        return np.nan

    features_by_country = {}
    for country, country_df in app_df.groupby("country", dropna=True):
        features_by_country[country] = build_country_feature_dict_from_df(country_df)

    countries = sorted(features_by_country.keys())
    if len(countries) < 2:
        return np.nan

    distances = []
    for ga, gb in combinations(countries, 2):
        dist = normalized_l1_distance(features_by_country[ga], features_by_country[gb])
        distances.append(dist)

    return float(np.mean(distances)) if distances else np.nan


# Load and prepare dataset
df = pd.read_csv(metrics_results, low_memory=False)
print("Raw shape:", df.shape)

if "country" in df.columns:
    df["country"] = df["country"].astype(str).str.strip().str.lower()

if "country_label" not in df.columns and "country" in df.columns:
    df["country_label"] = df["country"].map(COUNTRY_LABEL_MAP)

if "region" not in df.columns and "country" in df.columns:
    df["region"] = df["country"].map(REGION_MAP)

if "categories" in df.columns:
    df["category"] = df["categories"].fillna("Unknown").astype(str).str.strip()
elif "category" in df.columns:
    df["category"] = df["category"].fillna("Unknown").astype(str).str.strip()
else:
    df["category"] = "Unknown"

metric_candidates = [
    "app_country_ADII", "app_country_PCLR", "DGI",
    "pre_all_type_frequency_in_log", "post_all_type_frequency_in_log",
    "app_country_pre_sensitive_instances", "app_country_total_sensitive_instances",
    "observed_count", "disclosed_count", "missing_count", "misleading_count",
    "num_permissions", "num_dangerous_permissions", "num_trackers",
    "downloads_int", "average_score"
]
for col in metric_candidates:
    if col in df.columns:
        df[col] = coerce_numeric(df[col])

for col in ["pre_type_frequencies", "post_type_frequencies"]:
    if col in df.columns:
        df[col + "_map"] = df[col].apply(parse_semicolon_frequency_map)

for col in ["pre_domains", "post_domains"]:
    if col in df.columns:
        df[col + "_set"] = df[col].apply(parse_domain_set)

set_like_cols = [
    "pre_observed_data_types",
    "post_observed_data_types",
    "observed_set",
    "disclosed_set",
    "missing_set",
    "misleading_set",
    "data_safety_data_shared",
    "data_safety_data_collected",
    "pre_PII", "post_PII", "pre_PHI", "post_PHI", "pre_OTHER", "post_OTHER",
    # NOTE: the 12 privacy-policy disclosure columns below were previously
    # missing from this list. Since the fast path a few cells down only
    # treats a column as part of DISCLOSURE_COLUMNS if a matching "<col>_set"
    # column already exists, omitting them silently limited "disclosed" to
    # data_safety_data_shared/data_safety_data_collected only -- the twelve
    # privacy-policy-derived fields (e.g. what the policy says it collects)
    # were never counted as disclosed, which inflated DGI (the disclosure
    # gap) by counting policy-covered items as "missing".
    "privacy_policy_data_collection_disclosed_detail",
    "privacy_policy_collection_method_completeness_detail",
    "privacy_policy_data_sharing_disclosed_detail",
    "privacy_policy_sharing_purpose_completeness_detail",
    "privacy_policy_data_recipients_identified_detail",
    "privacy_policy_user_rights_completeness_detail",
    "privacy_policy_encryption_in_transit_status_detail",
    "privacy_policy_security_measures_completeness_detail",
    "privacy_policy_contact_provided_detail",
    "privacy_policy_childrens_data_completeness_detail",
    "privacy_policy_laws_named_detail",
    "privacy_policy_clarity_rating_detail",
]
for col in set_like_cols:
    if col in df.columns:
        df[col + "_set"] = df[col].apply(parse_listish_to_set)

if "app_country_ADII" in df.columns:
    df["ADII"] = df["app_country_ADII"]
if "app_country_PCLR" in df.columns:
    df["PCLR"] = df["app_country_PCLR"]

DISCLOSURE_COLUMNS = [
    "data_safety_data_shared",
    "data_safety_data_collected",
    "privacy_policy_data_collection_disclosed_detail", "privacy_policy_collection_method_completeness_detail", "privacy_policy_data_sharing_disclosed_detail", "privacy_policy_sharing_purpose_completeness_detail", "privacy_policy_data_recipients_identified_detail", "privacy_policy_user_rights_completeness_detail",
    "privacy_policy_encryption_in_transit_status_detail", "privacy_policy_security_measures_completeness_detail", "privacy_policy_contact_provided_detail", "privacy_policy_childrens_data_completeness_detail", "privacy_policy_laws_named_detail", "privacy_policy_clarity_rating_detail",
]

OBSERVED_COLUMNS = [
    "pre_observed_data_types", "post_observed_data_types",
    "pre_PII", "post_PII", "pre_PHI", "post_PHI", "pre_OTHER", "post_OTHER",
]

# Reuse parsed set columns when available to avoid reparsing every row.
observed_set_cols = [f"{c}_set" for c in OBSERVED_COLUMNS if f"{c}_set" in df.columns]
disclosed_set_cols = [f"{c}_set" for c in DISCLOSURE_COLUMNS if f"{c}_set" in df.columns]

assert len(disclosed_set_cols) == len(DISCLOSURE_COLUMNS), (
    "Expected a parsed '_set' column for every entry in DISCLOSURE_COLUMNS; "
    f"got {len(disclosed_set_cols)}/{len(DISCLOSURE_COLUMNS)}. Check that "
    "set_like_cols above includes every DISCLOSURE_COLUMNS entry."
)

if observed_set_cols and disclosed_set_cols:
    observed_sets = []
    disclosed_sets = []
    missing_sets = []
    misleading_sets = []

    for _, row in df.iterrows():
        observed = set().union(*[row[c] for c in observed_set_cols if isinstance(row[c], set)])
        disclosed = set().union(*[
            {tok for tok in row[c] if not is_invalid_disclosure_token(tok)}
            for c in disclosed_set_cols if isinstance(row[c], set)
        ])
        missing = observed - disclosed
        misleading = disclosed - observed

        observed_sets.append(sorted(observed))
        disclosed_sets.append(sorted(disclosed))
        missing_sets.append(sorted(missing))
        misleading_sets.append(sorted(misleading))

    dgi_df = pd.DataFrame({
        "observed_set_final": observed_sets,
        "disclosed_set_final": disclosed_sets,
        "missing_set": missing_sets,
        "misleading_set": misleading_sets,
    }, index=df.index)
    dgi_df["observed_count"] = dgi_df["observed_set_final"].str.len()
    dgi_df["disclosed_count"] = dgi_df["disclosed_set_final"].str.len()
    dgi_df["missing_count"] = dgi_df["missing_set"].str.len()
    dgi_df["misleading_count"] = dgi_df["misleading_set"].str.len()
    dgi_df["DGI"] = np.where(
        dgi_df["observed_count"].gt(0),
        dgi_df["missing_count"] / dgi_df["observed_count"],
        np.nan,
    )
else:
    dgi_df = df.apply(
        lambda row: compute_dgi_row(row, OBSERVED_COLUMNS, DISCLOSURE_COLUMNS),
        axis=1
    )

df = df.drop(
    columns=[
        "DGI",
        "observed_count",
        "disclosed_count",
        "missing_count",
        "misleading_count",
    ],
    errors="ignore"
)

df = pd.concat([df, dgi_df], axis=1)

valid_obs = df["observed_count"] > 0
if valid_obs.any():
    check = (
        df.loc[valid_obs, "DGI"].round(10)
        == (df.loc[valid_obs, "missing_count"] / df.loc[valid_obs, "observed_count"]).round(10)
    )
    print("DGI sanity check passed:", bool(check.all()))

if "app_id" not in df.columns:
    raise ValueError("Missing required column: 'app_id'")
if "country" not in df.columns:
    raise ValueError("Missing required column: 'country'")

app_as = (
    df.groupby("app_id", dropna=False)
      .apply(compute_app_level_as, include_groups=False)
      .rename("AS")
      .reset_index()
)

df = df.drop(columns=["AS"], errors="ignore").merge(app_as, on="app_id", how="left")

required_metrics = ["ADII", "DGI", "PCLR"]
missing_required = [c for c in required_metrics if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required metric columns after preparation: {missing_required}")

analysis_df = df.copy()
analysis_df = analysis_df.dropna(subset=["ADII", "DGI", "PCLR"]).copy()

analysis_df = analysis_df[
    analysis_df["region"].notna() & analysis_df["country_label"].notna()
].copy()

analysis_df.to_csv("../data/mhealth_apps_metrics.csv", index=False)

print("Analysis shape:", analysis_df.shape)
print("Unique apps:", analysis_df["app_id"].nunique())

display_cols = [
    "app_id", "country", "country_label", "region", "category",
    "ADII", "DGI", "PCLR", "AS",
    "observed_count", "disclosed_count", "missing_count", "misleading_count"
]
display_cols = [c for c in display_cols if c in analysis_df.columns]


## 3. Quality checks and metric coverage


In [ ]:

summary = pd.DataFrame({
    "non_null": analysis_df[["ADII", "DGI", "PCLR", "AS"]].notna().sum(),
    "mean": analysis_df[["ADII", "DGI", "PCLR", "AS"]].mean(),
    "median": analysis_df[["ADII", "DGI", "PCLR", "AS"]].median(),
    "std": analysis_df[["ADII", "DGI", "PCLR", "AS"]].std(),
    "min": analysis_df[["ADII", "DGI", "PCLR", "AS"]].min(),
    "max": analysis_df[["ADII", "DGI", "PCLR", "AS"]].max(),
}).round(4)
summary



## 4. App-level aggregation for stable app-centric analysis

Since the dataset may contain multiple observations per app across countries, we compute:
- a **country-level view** for geographic comparisons
- an **app-level summary** for cross-app comparisons


In [ ]:

country_level = analysis_df.copy()

# Rebuild app-level metrics from the prepared analysis dataframe.
# This keeps the notebook stable even after partial reruns.
metric_agg = {
    "ADII": "mean",
    "DGI": "mean",
    "PCLR": "mean",
    "AS": "mean",
}

meta_agg = {
    "country_label": "nunique",
    "region": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "category": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
    "num_permissions": "mean",
    "num_dangerous_permissions": "mean",
    "num_trackers": "mean",
    "downloads_int": "mean",
    "average_score": "mean",
}

agg_dict = {}
for k, v in {**metric_agg, **meta_agg}.items():
    if k in analysis_df.columns:
        agg_dict[k] = v

app_level = analysis_df.groupby("app_id", as_index=False).agg(agg_dict)
app_level = app_level.rename(columns={"country_label": "country_count"})
app_level_all = app_level.copy()

# Defensive fallback: if PCLR is absent here but exists in analysis_df under an older name,
# rebuild it so downstream summary cells do not fail.
if "PCLR" not in app_level_all.columns:
    if "app_country_PCLR" in analysis_df.columns:
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)["app_country_PCLR"]
            .mean()
            .rename(columns={"app_country_PCLR": "PCLR"})
        )
        app_level_all = app_level_all.merge(pclr_fallback, on="app_id", how="left")
    elif {"app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"}.issubset(analysis_df.columns):
        pclr_fallback = (
            analysis_df.groupby("app_id", as_index=False)[
                ["app_country_pre_sensitive_instances", "app_country_total_sensitive_instances"]
            ]
            .mean()
        )
        denom = pclr_fallback["app_country_total_sensitive_instances"].replace(0, np.nan)
        pclr_fallback["PCLR"] = pclr_fallback["app_country_pre_sensitive_instances"] / denom
        app_level_all = app_level_all.merge(pclr_fallback[["app_id", "PCLR"]], on="app_id", how="left")

print("Country-level rows:", len(country_level))
print("App-level rows:", len(app_level_all))
print("App-level metric columns:", [c for c in ["ADII", "DGI", "PCLR", "AS"] if c in app_level_all.columns])
app_level = app_level_all.copy()



In [ ]:
print("Before shape:", df.shape)
print("After shape:", analysis_df.shape)

# -----------------------------
# Unique app counts per country
# -----------------------------
country_app_counts = (
    analysis_df.groupby(["region", "country_label"])["app_id"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index(name="unique_app_count")
)

country_app_counts